# Giai đoạn 2 trên Kaggle (2 × T4)

Mỗi lần chạy notebook = **một seed**: train song song `sched` (GPU 0) và `sched_bc` (GPU 1), rồi đánh giá các checkpoint của chính lần chạy đó.

**Trước khi chạy:**
1. *Settings → Accelerator:* **GPU T4 x2**.
2. *Settings → Internet:* **On** (cần xác minh số điện thoại).
3. Sửa `SEED` ở ô tiếp theo (1, 2 hoặc 3 — mỗi seed một lần chạy).
4. Bấm **Save Version → Save & Run All (Commit)** để notebook chạy nền (tối đa 12 giờ), có thể đóng trình duyệt.

Kết quả: file `phase2_results-*.tgz` trong tab **Output** của version đó.

In [ ]:
SEED = 1            # 1, 2 hoặc 3
ITERS = 1000        # số iteration mỗi lần train
SMOKE_ONLY = False  # True: chỉ kiểm tra cài đặt và đo tốc độ, không train

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
%%bash
set -e
pip install -q uv
rm -rf /tmp/mjlab
git clone -q --depth 1 -b research/amp-velocity https://github.com/nghiatran0106/mjlab.git /tmp/mjlab
cd /tmp/mjlab
git log --oneline -1
UV_LINK_MODE=copy uv sync --locked --no-dev --extra cu128 > /tmp/uv_sync.log 2>&1 || { tail -20 /tmp/uv_sync.log; exit 1; }
uv run --frozen --no-dev --extra cu128 python -c "import torch, warp as wp; wp.init(); print('torch', torch.__version__, '| GPUs:', torch.cuda.device_count(), torch.cuda.get_device_name(0), '| warp CUDA:', wp.is_cuda_available())"

## Kiểm tra nhanh và đo tốc độ (khoảng 5 phút)
Train 10 iteration với 4096 môi trường. Nếu ô này lỗi, dừng lại và gửi log cho Claude.

In [ ]:
%%bash
cd /tmp/mjlab
export MUJOCO_GL=egl
timeout 1500 uv run --frozen --no-dev --extra cu128 train Mjlab-Velocity-Flat-Unitree-G1 \
  --gpu-ids '[0]' --env.scene.num-envs 4096 --agent.max-iterations 10 \
  --agent.logger tensorboard --agent.upload-model False --agent.run-name kaggle-smoke \
  > /tmp/smoke.log 2>&1 || { tail -30 /tmp/smoke.log; exit 1; }
sed 's/\x1b\[[0-9;]*m//g' /tmp/smoke.log | grep -E "Iteration time" | tail -3

In [ ]:
import re, subprocess
log = open('/tmp/smoke.log').read()
times = [float(t) for t in re.findall(r'Iteration time: ([0-9.]+)s', log)]
if times:
    t = sorted(times)[len(times) // 2]
    # Two runs share nothing (one per GPU); expert data collection and evaluation add ~20-30 min.
    print(f'~{t:.2f} s/iteration -> {ITERS} iterations ≈ {t * ITERS / 3600:.1f} h (+ ~0.5 h data/eval)')
    if t * ITERS / 3600 > 10.5:
        print('CẢNH BÁO: vượt giới hạn 12 giờ của Kaggle — giảm ITERS.')

## Chạy Giai đoạn 2 cho seed đã chọn

In [ ]:
import os, subprocess
assert not SMOKE_ONLY, 'SMOKE_ONLY=True: dừng ở đây.'
cmd = (f'cd /tmp/mjlab && SEEDS="{SEED}" GPUS="0 1" MAX_ITERATIONS={ITERS} MUJOCO_GL=egl '
       f'bash src/mjlab/tasks/velocity_amp/scripts/run_phase2.sh 2>&1 '
       f'| tee /kaggle/working/phase2_seed{SEED}.log')
proc = subprocess.run(['bash', '-c', cmd + ' | grep -E "^==|^\\||Done|rror|Traceback"'])
print('exit', proc.returncode)

In [ ]:
%%bash
cp /tmp/mjlab/logs/phase2_results-*.tgz /kaggle/working/ 2>/dev/null || echo "Không có gói kết quả — xem phase2_seed*.log"
cp /tmp/mjlab/logs/phase2_summary.md /kaggle/working/ 2>/dev/null || true
ls -la /kaggle/working/
cat /kaggle/working/phase2_summary.md 2>/dev/null || true